# OLS Regression: polars_reg vs R Verification

This notebook verifies that `polars_reg.ols()` produces identical results to R's `lm()` across
several standard error types (iid, HC1, HC2, HC3) using the **swiss** dataset (47 observations).

In [ ]:
import sys
import tempfile
from pathlib import Path

# Ensure polars_reg and the helper are importable
REPO = Path.home() / "research" / "polars_reg"
sys.path.insert(0, str(REPO))
sys.path.insert(0, str(REPO / "notebooks" / "verification"))

import polars as pl
import polars_reg as pr
import r_helper

In [ ]:
# Load the swiss dataset from R and rename dotted columns
df = r_helper.load_r_dataset(
    "swiss",
    extra_code='names(df) <- gsub("\\\\.", "_", names(df))'
)
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns}")
df.head(5)

In [ ]:
# Save CSV for R scripts
csv_path = tempfile.mktemp(suffix=".csv")
df.to_pandas().to_csv(csv_path, index=False)
print(f"CSV saved to: {csv_path}")

## 1. Basic OLS (iid standard errors)

In [ ]:
result_iid = pr.ols("Fertility ~ Agriculture + Education + Catholic", data=df)
print(f"N={result_iid.n_obs}, R2={result_iid.r_squared:.6f}")

In [ ]:
r_script = f'''
df <- read.csv("{csv_path}")
model <- lm(Fertility ~ Agriculture + Education + Catholic, data=df)
vcov_mat <- vcov(model)
{r_helper.R_EXTRACT}
'''
r_iid = r_helper.run_r_regression(r_script)
r_helper.compare(result_iid, r_iid, rtol=1e-8, label="OLS iid")

## 2. OLS with HC1 robust standard errors

In [ ]:
result_hc1 = pr.ols("Fertility ~ Agriculture + Education + Catholic", data=df, vcov="HC1")
print(f"N={result_hc1.n_obs}, R2={result_hc1.r_squared:.6f}")

In [ ]:
r_script = f'''
library(sandwich)
df <- read.csv("{csv_path}")
model <- lm(Fertility ~ Agriculture + Education + Catholic, data=df)
vcov_mat <- vcovHC(model, type="HC1")
{r_helper.R_EXTRACT}
'''
r_hc1 = r_helper.run_r_regression(r_script)
r_helper.compare(result_hc1, r_hc1, rtol=1e-6, label="OLS HC1")

## 3. OLS with HC2 robust standard errors

In [ ]:
result_hc2 = pr.ols("Fertility ~ Agriculture + Education + Catholic", data=df, vcov="HC2")
print(f"N={result_hc2.n_obs}, R2={result_hc2.r_squared:.6f}")

In [ ]:
r_script = f'''
library(sandwich)
df <- read.csv("{csv_path}")
model <- lm(Fertility ~ Agriculture + Education + Catholic, data=df)
vcov_mat <- vcovHC(model, type="HC2")
{r_helper.R_EXTRACT}
'''
r_hc2 = r_helper.run_r_regression(r_script)
r_helper.compare(result_hc2, r_hc2, rtol=1e-6, label="OLS HC2")

## 4. OLS with HC3 robust standard errors

In [ ]:
result_hc3 = pr.ols("Fertility ~ Agriculture + Education + Catholic", data=df, vcov="HC3")
print(f"N={result_hc3.n_obs}, R2={result_hc3.r_squared:.6f}")

In [ ]:
r_script = f'''
library(sandwich)
df <- read.csv("{csv_path}")
model <- lm(Fertility ~ Agriculture + Education + Catholic, data=df)
vcov_mat <- vcovHC(model, type="HC3")
{r_helper.R_EXTRACT}
'''
r_hc3 = r_helper.run_r_regression(r_script)
r_helper.compare(result_hc3, r_hc3, rtol=1e-6, label="OLS HC3")

## 5. Full model with all regressors

In [ ]:
result_full = pr.ols(
    "Fertility ~ Agriculture + Examination + Education + Catholic + Infant_Mortality",
    data=df,
)
print(f"N={result_full.n_obs}, R2={result_full.r_squared:.6f}")

In [ ]:
r_script = f'''
df <- read.csv("{csv_path}")
model <- lm(Fertility ~ Agriculture + Examination + Education + Catholic + Infant_Mortality, data=df)
vcov_mat <- vcov(model)
{r_helper.R_EXTRACT}
'''
r_full = r_helper.run_r_regression(r_script)
r_helper.compare(result_full, r_full, rtol=1e-8, label="Full model iid")

## 6. Model without intercept

In [ ]:
result_noint = pr.ols(
    "Fertility ~ Agriculture + Education + Catholic - 1",
    data=df,
)
print(f"N={result_noint.n_obs}, names={result_noint.names}")

In [ ]:
r_script = f'''
df <- read.csv("{csv_path}")
model <- lm(Fertility ~ Agriculture + Education + Catholic - 1, data=df)
vcov_mat <- vcov(model)
{r_helper.R_EXTRACT}
'''
r_noint = r_helper.run_r_regression(r_script)
r_helper.compare(result_noint, r_noint, rtol=1e-8, label="No intercept")

## 7. Summary display

In [ ]:
print(result_full.summary())

In [ ]:
result_full.coef_table()

## Summary

All OLS tests compare `polars_reg` against R's `lm()` + `sandwich` package:

| Test | SE type | Tolerance | Status |
|------|---------|-----------|--------|
| Basic OLS | iid | 1e-8 | see above |
| Robust HC1 | HC1 | 1e-6 | see above |
| Robust HC2 | HC2 | 1e-6 | see above |
| Robust HC3 | HC3 | 1e-6 | see above |
| Full model | iid | 1e-8 | see above |
| No intercept | iid | 1e-8 | see above |